# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [2]:
!pip -q install duckdb pandas pyarrow huggingface_hub

In [3]:
import duckdb
import pandas as pd
from huggingface_hub import login

login(token=HF_TOKEN)

## 1. Unit of Analysis + Time Window

**Unit of Analysis (One Row):**

One row represents the daily performance of a single content item for a specific client on a specific report date.

**Time Window:**

For this assignment, I use data from **March 2026 (2026-03)** as the analysis window, following the assignment recommendation to work on a mid-panel month.

## 2. Fields: Feature / Label / Context / Excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_ctr
- gsc_position
- gsc_data_available

### Label (Example)
- Future daily clicks (or future performance score)

### Context
- report_date
- client_hash_id
- content_hash_id

### Excluded
- Future information (future clicks, future impressions, future rankings)

**Reason:** These fields would leak future information into the model and produce unrealistically high performance.

In [6]:
!pip -q install duckdb pandas pyarrow huggingface_hub

In [8]:
import duckdb
import pandas as pd
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

con = duckdb.connect()

In [11]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

In [10]:
from huggingface_hub import list_repo_files
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print(files[:20])

['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet', 'fact_content_daily_performance/month=2025-07/data_0.parquet', 'fact_content_daily_performance/month=2025-08/data_0.parquet', 'fact_content_daily_performance/month=2025-09/data_0.parquet', 'fact_content_daily_performance/month=2025-10/data_0.parquet', 'fact_content_daily_performance/month=2025-11/data_0.parquet', 'fact_content_daily_performance/month=2025-12/data_0.parquet', 'fact_content_daily_performance/month=2026-01/data_0.parquet', 'fact_content_daily_performance/month=2026-02/data_0.parquet', 'fact_content_daily_performance/month=20

In [15]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS c
FROM {REL}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [14]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {REL};
"""

con.sql(query).df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [16]:
con.sql("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


# Verification Summary

- The grain query returned no duplicate rows, confirming the unit of analysis.
- The count query verified the number of rows and date range for March 2026.
- The availability query showed how many rows contain valid Google Search Console data using `IS TRUE`.

## 2. Fields: Feature / Label / Context / Excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_ctr
- gsc_position
- gsc_data_available

### Label (Example)
- Future daily clicks (or future performance score)

### Context
- report_date
- client_hash_id
- content_hash_id

### Excluded
- Future information (future clicks, future impressions, future rankings)

**Reason:** These fields would leak future information into the model and produce unrealistically high performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.